ERA5 reanalysis atmospheres
===========================

MCEq can run on a **measured or reanalysed atmosphere** instead of a
parametrization, through
`MCEq.geometry.density_profiles.TabulatedAtmosphere` and its location-centred
sibling `TabulatedLocationCentered`. They read a CSV table; converting a
dataset into that table is left to the user, because every archive has its own
API and file format and MCEq does not want to depend on any of them.

This notebook is the worked example for **ERA5**, the ECMWF reanalysis. It

1. turns the ERA5 **surface geopotential** into a table of ground elevations —
   the field that says where each atmospheric column actually stops,
2. converts an ERA5 netCDF download into MCEq tables — one per day, each a
   global longitude/latitude grid of vertical columns,
3. runs MCEq on them and compares against NRLMSISE-00,
4. maps and animates the result globally, with every plotted number coming
   out of the MCEq atmosphere interface rather than straight from the file,
5. shows why the grid matters for neutrinos: at large zenith angles, and for
   upgoing events in particular, the shower develops in a completely different
   column than the one above the detector.

> **Not executed when the documentation is built.** It needs a CDS account and
> a multi-hundred-MB download. Run it locally.

What you need
-------------

```bash
pip install cdsapi xarray netcdf4 cartopy
```

(cartopy downloads Natural Earth coastline data the first time it draws
a map, so the first plot needs a network connection.)

and, for the download, a CDS account with an API key in `~/.cdsapirc`
(register at <https://cds.climate.copernicus.eu>, then follow
<https://cds.climate.copernicus.eu/how-to-api> and accept the dataset licence
once from its download page).

The table format
----------------

A table is a CSV file. In its simplest form it is **one vertical column**:

```
# MCEq tabulated atmosphere v1
h_cm,T_K,p_hPa
283400.0,247.1,681.2
510000.0,231.4,500.0
900000.0,,300.0
```

* `h_cm` is required: height above sea level in cm.
* Density comes either from a `rho_gcm3` column directly, or from `T_K` and
  `p_hPa` via the dry-air ideal gas law.
* `T_K` also feeds `get_temperature()`, `p_hPa` feeds `get_pressure()`.
* Comment lines start with `#`, column order does not matter, unknown columns
  are ignored, rows may be in any order, and missing values are an empty field
  or `nan`.

Adding **`lat_deg` and `lon_deg`** turns it into a *grid* of columns — one row
per (grid node, level), long format:

```
lat_deg,lon_deg,h_cm,T_K,p_hPa
-90.0,0.0,110.9,242.8,1000
-90.0,0.0,1352.0,241.6,975
...
```

The nodes must form a regular longitude/latitude grid, and every column must
carry the same number of levels. That is exactly how ERA5 pressure-level data
is shaped, so the conversion is a reshape.

In [ ]:
import gzip
import os

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

G0 = 9.80665  # standard gravity, m/s^2

# --- input ----------------------------------------------------------------
# An ERA5 pressure-level download: "ERA5 daily statistics on pressure levels"
# (daily mean), variables *temperature* and *geopotential*, all 37 levels,
# global. The CDS portal delivers one netCDF file per variable, so point this
# at both; a single file holding both variables works just as well.
NC_FILES = [
    "geopotential_stream-oper_daily-mean.nc",
    "temperature_0_daily-mean.nc",
]

# The ground. ERA5's surface geopotential is one of its *invariant* fields --
# it is the model's fixed orography, so a single timestep covers the whole
# archive and this file never has to be downloaded again:
#
#   c.retrieve("reanalysis-era5-single-levels", {
#       "product_type": "reanalysis", "variable": ["geopotential"],
#       "year": "2013", "month": "08", "day": "09", "time": "12:00",
#       "data_format": "netcdf"}, OROGRAPHY_FILE)
#
# Beware the name collision: the *pressure-level* z in NC_FILES is the height
# of each pressure surface and moves with the weather; the *single-level* z
# here is the terrain and does not.
OROGRAPHY_FILE = "era5_orography.nc"

# How much of the native 0.25 deg grid to keep. What limits this is the size of
# the tables on disk, not MCEq: one column costs about 0.35 ms, but a CSV row
# costs ~46 bytes and there is one per level per column.
#
#   STRIDE  grid       columns    map      table (gzipped)
#       16  4 deg        4 140     1.4 s      1.7 MB / day
#        4  1 deg       65 160      24 s       26 MB / day
#        2  0.5 deg    259 920      93 s     ~100 MB / day
#        1  0.25 deg     1.04 M    ~6 min    ~410 MB / day
#
# The last two table sizes and the 0.25 deg timing are extrapolated; the cost
# per column is flat at 0.34-0.36 ms across every grid measured above.
#
# 1 deg is the sweet spot for a global figure: on a map this size it is already
# indistinguishable from 0.5 deg, so the finer grids cost 4x and 16x for
# nothing anyone can see. Drop STRIDE for a regional crop, where the impact
# point moves far less than a grid cell.
STRIDE = 4

OUTPUT_DIR = "era5_tables"

# --- the detector ---------------------------------------------------------
# KM3NeT/ARCA: off Capo Passero, and far enough from the pole that the azimuth
# angle genuinely changes which column the shower develops in.
SITE_NAME = "KM3NeT-ARCA"
SITE_LON, SITE_LAT = 16.1, 36.267
SITE_DEPTH_M = 3500.0
SITE_ELEVATION_M = 0.0

In [ ]:
# merge() aligns the per-variable files on their shared coordinates and fails
# loudly if the requests behind them did not use the same grid or dates.
ds = xr.merge(
    [xr.open_dataset(path, chunks={"valid_time": 1}) for path in NC_FILES],
    join="exact",
    compat="override",  # the files hold different variables; nothing to reconcile
)
print(ds)

HAS_GEOPOTENTIAL = "z" in ds.data_vars
print(f"\ngeopotential present: {HAS_GEOPOTENTIAL}")

## The ground

Everything else in this notebook sits on top of this field, so it comes first.

ERA5's pressure levels do not stop at the terrain: below it they are
**extrapolated**, and those levels are fictitious. Over the Antarctic plateau
the 1000 hPa surface comes out at $-55$ m, some 2.9 km *below* the ice. Nothing
in a pressure-level file marks where the ground is, so without this field every
column would have to start at sea level and the overburden over any mountain
would be badly wrong.

The surface geopotential fixes that, and it is cheap: it is an **invariant**
field — the model's own orography — so one timestep covers the whole of ERA5.
Converted to a height, $h_\mathrm{surface} = z_\mathrm{surface}/g_0$, it is
written here to its own small table, indexed by longitude and latitude just
like the atmosphere tables and read back with the same idea.

In [ ]:
oro = xr.open_dataset(OROGRAPHY_FILE)
if "time" in oro.dims:
    oro = oro.isel(time=0)  # invariant: any timestep will do

# Sample onto the same grid the atmosphere tables will use.
grid_lat = ds["latitude"].values[::STRIDE]
grid_lon = ds["longitude"].values[::STRIDE]

elevation_m = (
    (oro["z"] / G0)
    .interp(
        latitude=("latitude", grid_lat),
        longitude=("longitude", grid_lon),
        method="linear",
    )
    .values
)

SURFACE_TABLE = os.path.join(OUTPUT_DIR, "era5_surface.csv")
os.makedirs(OUTPUT_DIR, exist_ok=True)
with open(SURFACE_TABLE, "w") as out:
    out.write("# ERA5 surface elevation, from the invariant surface geopotential\n")
    out.write(f"# {grid_lat.size} latitudes x {grid_lon.size} longitudes\n")
    out.write("lat_deg,lon_deg,h_surface_m\n")
    np.savetxt(
        out,
        np.column_stack([
            np.repeat(grid_lat, grid_lon.size),
            np.tile(grid_lon, grid_lat.size),
            elevation_m.reshape(-1),
        ]),
        delimiter=",", fmt="%.4f,%.4f,%.2f",
    )

print(f"{SURFACE_TABLE}  {elevation_m.shape[0]}x{elevation_m.shape[1]} nodes")
print(f"elevation on the table grid  {elevation_m.min():8.1f} .. "
      f"{elevation_m.max():.1f} m")
print(f"at full 0.1 deg resolution   {float((oro['z'] / G0).min()):8.1f} .. "
      f"{float((oro['z'] / G0).max()):.1f} m")

### The surface elevation of the Earth

Plotted at ERA5's native resolution, which is finer than the grid the tables
use. The model orography is smoothed, so peaks come out lower than the real
summits — the South Pole node reads about 2765 m against IceCube's 2835 m, and
the Himalaya tops out near 6000 m rather than 8850 m. For a detector that
matters, use the site's surveyed elevation rather than the reanalysis value.

In [ ]:
import warnings

import cartopy.crs as ccrs
import matplotlib as mpl
from cartopy.util import add_cyclic_point

# Cartopy clips coastline polygons at the edge of the Robinson projection and
# shapely grumbles about the empty geometries that produces. Harmless, loud.
warnings.filterwarnings(
    "ignore", message="invalid value encountered in create_collection"
)


def paled(name, amount=0.55):
    """A washed-out version of a colormap, for use as a backdrop.

    Drawing the mesh semi-transparent instead would be the obvious way, but
    add_cyclic_point's wrapped column lands back on longitude 0 in a Robinson
    projection and any alpha < 1 makes that overlap show up as a seam down the
    middle of the map. Fading the colours and staying opaque avoids it.
    """
    colors = mpl.colormaps[name](np.linspace(0.0, 1.0, 256))
    colors[:, :3] = 1.0 - amount * (1.0 - colors[:, :3])
    return mpl.colors.ListedColormap(colors)


def global_map(ax, field, lat, lon, cmap, label, shrink=0.85, **kwargs):
    """Draws a lon/lat field on a Robinson projection."""
    # add_cyclic_point insists the longitude axis be equally spaced, which a
    # float32 axis is not, quite: ERA5's 0.2 deg steps come back as 0.199982 to
    # 0.200012. Rebuild it from its endpoints when it is uniform to that sort
    # of tolerance, and leave it alone -- to raise -- when it genuinely is not.
    lon = np.asarray(lon, dtype=float)
    steps = np.diff(lon)
    if np.ptp(steps) < 1e-3 * np.abs(steps.mean()) * len(lon):
        lon = np.linspace(lon[0], lon[-1], lon.size)
    cyclic, lon_c = add_cyclic_point(field, coord=lon)
    mesh = ax.pcolormesh(
        lon_c, lat, cyclic, transform=ccrs.PlateCarree(),
        cmap=cmap, shading="auto", **kwargs
    )
    ax.coastlines(lw=0.4, color="0.3")
    ax.set_global()
    plt.colorbar(mesh, ax=ax, orientation="horizontal", pad=0.04, label=label,
                 shrink=shrink, aspect=40, extend="both")
    return mesh


# Every second point: 0.2 deg is plenty for a global figure.
fine = (oro["z"] / G0).isel(latitude=slice(None, None, 2), longitude=slice(None, None, 2))

fig = plt.figure(figsize=(9, 5.0), dpi=120)
ax = fig.add_subplot(1, 1, 1, projection=ccrs.Robinson())
global_map(ax, fine.values, fine["latitude"].values, fine["longitude"].values,
           "terrain", "surface elevation [m]", vmin=-500, vmax=5500)
ax.plot(SITE_LON, SITE_LAT, "r*", ms=12, transform=ccrs.PlateCarree())
ax.set_title("ERA5 surface geopotential / $g_0$")
fig.subplots_adjust(left=0.04, right=0.96, top=0.92)

### Heights

MCEq integrates along a slant path through *geometric height*, so every level
needs one. Geopotential gives it directly:

$$ h = \frac{z}{g_0}, \qquad g_0 = 9.80665\ \mathrm{m/s^2} $$

which is why `geopotential` belongs in the CDS request next to `temperature`.
It is what makes the columns differ *geometrically* and not just thermally: in
this download the 1 hPa surface sits at 41.2 km over the South Pole and at
48.7 km over the Mediterranean.

Two properties of these heights are worth knowing before using them.

* They are **geopotential** heights, i.e. $z/g_0$. Geometric height differs by
  the variation of gravity with altitude, which is +0.3 % at 20 km and +0.8 %
  at 50 km. MCEq's own geometry is spherical, so this is well inside the
  accuracy of treating the atmosphere as horizontally layered.
* Below the terrain ERA5 **extrapolates**, and those levels are fictitious —
  over the Antarctic plateau the 1000 hPa surface comes out at $-55$ m, some
  2.9 km below the ice. Nothing in the pressure-level file marks where the
  ground is, so `TabulatedAtmosphere` cannot find it: left to itself it starts
  at the table's lowest non-negative height, i.e. essentially sea level. **For
  a site on high ground, pass `surface_elevation_m`** and the integration
  starts there instead.

**Without geopotential** the heights have to be rebuilt from the temperatures
with the hypsometric equation,

$$ \Delta z = \frac{R_d}{g_0}\,\bar{T}\,\ln\frac{p_\mathrm{low}}{p_\mathrm{up}}, $$

integrated upward from an anchor. The *thicknesses* are good — they use the
real temperatures — but the anchor is a guess: the fallback below puts the
1000 hPa surface at its US-Standard height of 111 m everywhere, which is a
rigid vertical shift of the whole column and flattens exactly the structure the
map further down is meant to show. The converter uses geopotential when present
and falls back to the integration only when it is not.

In [ ]:
R_D = 287.06  # gas constant of dry air, J/(kg K)
H_1000_HPA_M = 110.9  # US Standard height of the 1000 hPa surface


def column_heights(pressure_hpa, temperature_k, geopotential=None):
    """Geometric height of every pressure level, in metres.

    Args:
      pressure_hpa: (n_lev,) levels, ascending in height (descending pressure)
      temperature_k: (..., n_lev) temperatures
      geopotential: (..., n_lev) geopotential in m^2/s^2, or None

    Returns:
      (..., n_lev) heights above sea level in metres
    """
    if geopotential is not None:
        return geopotential / G0

    # Hypsometric integration upward from the bottom level.
    thickness = (
        R_D
        / G0
        * 0.5
        * (temperature_k[..., 1:] + temperature_k[..., :-1])
        * np.log(pressure_hpa[:-1] / pressure_hpa[1:])
    )
    base = np.full(temperature_k.shape[:-1] + (1,), H_1000_HPA_M)
    return np.concatenate([base, H_1000_HPA_M + np.cumsum(thickness, axis=-1)], axis=-1)


def write_gridded_table(filename, lat, lon, h_cm, t_k, p_hpa, comment):
    """Writes a v1 MCEq tabulated-atmosphere CSV holding a grid of columns.

    Args:
      lat: (n_lat,) latitudes, lon: (n_lon,) longitudes
      h_cm, t_k: (n_lat, n_lon, n_lev); p_hpa: (n_lev,)
    """
    n_lat, n_lon, n_lev = h_cm.shape
    lat_col = np.repeat(lat, n_lon * n_lev)
    lon_col = np.tile(np.repeat(lon, n_lev), n_lat)
    p_col = np.tile(p_hpa, n_lat * n_lon)
    rows = np.column_stack(
        [lat_col, lon_col, h_cm.reshape(-1), t_k.reshape(-1), p_col]
    )
    opener = gzip.open if filename.endswith(".gz") else open
    with opener(filename, "wt") as out:
        out.write("# MCEq tabulated atmosphere v1\n")
        out.write(f"# {comment}\n")
        out.write("lat_deg,lon_deg,h_cm,T_K,p_hPa\n")
        np.savetxt(out, rows, delimiter=",", fmt="%.4f,%.4f,%.6e,%.3f,%.6g")
    return filename

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

sub = ds.isel(latitude=slice(None, None, STRIDE), longitude=slice(None, None, STRIDE))
lat = sub["latitude"].values
lon = sub["longitude"].values

# Order the levels by ascending height, i.e. descending pressure.
p_hpa = np.sort(sub["pressure_level"].values)[::-1]

table_files = []
for step in range(sub.sizes["valid_time"]):
    day = sub.isel(valid_time=step)
    date = str(day["valid_time"].values)[:10]

    t_k = (
        day["t"]
        .sel(pressure_level=p_hpa)
        .transpose("latitude", "longitude", "pressure_level")
        .values.astype(np.float64)
    )
    z = (
        day["z"]
        .sel(pressure_level=p_hpa)
        .transpose("latitude", "longitude", "pressure_level")
        .values.astype(np.float64)
        if HAS_GEOPOTENTIAL
        else None
    )
    h_cm = column_heights(p_hpa, t_k, z) * 1e2

    path = write_gridded_table(
        os.path.join(OUTPUT_DIR, f"era5_{date}.csv.gz"),
        lat, lon, h_cm, t_k, p_hpa,
        f"ERA5 daily mean, {date}, {lat.size}x{lon.size} grid, "
        f"heights from {'geopotential' if HAS_GEOPOTENTIAL else 'hypsometric integration'}",
    )
    table_files.append(path)
    print(f"{path}  {os.path.getsize(path) / 1e6:5.1f} MB  "
          f"top level {h_cm[..., -1].mean() / 1e5:.1f} km (mean)")

### One site

`("Tabulated", (path, coord))` picks a single column out of the grid;
`("Tabulated_LC", (path, coord, depth_m))` binds the grid to a detector and
samples it at the shower impact point. Start with the plain column and compare
it to NRLMSISE-00 at the same place and season.

In [ ]:
import MCEq.geometry.density_profiles as dp

table = dp.load_atmosphere_table(table_files[0])
print(table)


def load_surface_table(filename):
    """Reads the surface table written above back into a grid.

    Returns:
      (lat_axis, lon_axis, elevation) with elevation shaped (n_lat, n_lon)
    """
    # Strip full-line comments first: numpy's names=True takes the *first*
    # line as the header whether it is a comment or not.
    with open(filename) as handle:
        rows = [line for line in handle if line.strip()
                and not line.lstrip().startswith("#")]
    data = np.genfromtxt(rows, delimiter=",", names=True, dtype=float)
    lat_axis, lon_axis = np.unique(data["lat_deg"]), np.unique(data["lon_deg"])
    elevation = np.full((lat_axis.size, lon_axis.size), np.nan)
    elevation[
        np.searchsorted(lat_axis, data["lat_deg"]),
        np.searchsorted(lon_axis, data["lon_deg"]),
    ] = data["h_surface_m"]
    if np.isnan(elevation).any():
        raise ValueError(f"{filename} does not cover a full grid")
    return lat_axis, lon_axis, elevation


surface_lat, surface_lon, surface_m = load_surface_table(SURFACE_TABLE)


def elevation_for(table):
    """Surface elevation re-indexed onto *table*'s own axes.

    Both grids hold the same nodes, but nothing guarantees the same order --
    ERA5 counts latitude north-to-south, while load_atmosphere_table sorts it
    south-to-north. Matching on the coordinates rather than trusting the
    positions is what keeps the Antarctic plateau out of the Arctic Ocean.
    """
    jj = [int(np.argmin(np.abs(surface_lat - value))) for value in table.lat_deg]
    ii = [int(np.argmin(np.abs(surface_lon - value))) for value in table.lon_deg]
    return surface_m[np.ix_(jj, ii)]


table_elevation_m = elevation_for(table)
print(f"surface under the grid  {table_elevation_m.min():.1f} .. "
      f"{table_elevation_m.max():.1f} m")

era5_atm = dp.TabulatedAtmosphere(
    table, coord=(SITE_LON, SITE_LAT), location=SITE_NAME, season="July"
)
msis_atm = dp.MSIS00LocationCentered(
    detector_coord=(SITE_LON, SITE_LAT), depth_m=SITE_DEPTH_M, season="July"
)
for atm in (era5_atm, msis_atm):
    atm.set_theta(0.0)
print(f"ERA5 vertical column {era5_atm.max_X:8.2f} g/cm^2")
print(f"MSIS vertical column {msis_atm.max_X:8.2f} g/cm^2")

#### Profiles

Geopotential gives every level a real height, so the natural vertical
coordinate is altitude. The sampling is still done in **slant depth** — `X` is
what MCEq integrates in, and `X2h` maps it back — so the curves are read out of
exactly the interface the solver uses: `X2rho`, `X2h`, `get_temperature`.

In [ ]:
X = np.geomspace(1.0, min(era5_atm.max_X, msis_atm.max_X) * 0.999, 300)

fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), dpi=120, sharey=True)


def profile(atm, X):
    """rho, T and height along a slant-depth grid, all via the interface."""
    h_cm = atm.X2h(X)
    temp = np.array([float(atm.get_temperature(h)) for h in h_cm])
    return atm.X2rho(X), temp, h_cm / 1e5


rho_e, t_e, h_e = profile(era5_atm, X)
rho_m, t_m, h_m = profile(msis_atm, X)

axes[0].plot(rho_e, h_e, "-", label="ERA5")
axes[0].plot(rho_m, h_m, "--", label="MSIS00")
axes[1].plot(t_e, h_e, "-", label="ERA5")
axes[1].plot(t_m, h_m, "--", label="MSIS00")

# Compared at equal depth, which is where the two models actually meet: the
# same X is the same overburden, whatever height each puts it at.
axes[2].plot(rho_e / rho_m, h_e, "-", label=r"$\rho$")
axes[2].plot(t_e / t_m, h_e, "--", label="T")
axes[2].axvline(1.0, color="0.6", lw=0.8)

# The geometric offset between the two is a difference, not a ratio: both
# heights go to zero at the ground, where a ratio says nothing.
off = axes[2].twiny()
off.plot((h_e - h_m) * 1e3, h_e, ":", color="tab:green", label="h")
off.set_xlabel("ERA5 - MSIS00 height at equal $X$ [m]", fontsize=9)
off.tick_params(labelsize=8)

axes[0].set_xscale("log")
axes[0].set_xlabel(r"$\rho$ [g/cm$^3$]")
axes[1].set_xlabel("T [K]")
axes[2].set_xlabel(r"ERA5 / MSIS00 at equal $X$ ($\rho$, T)")
axes[0].set_ylabel("height [km]")
for ax in axes:
    ax.grid(alpha=0.3)
    ax.legend()
fig.suptitle(f"{SITE_NAME}, {str(ds['valid_time'].values[0])[:10]}")
fig.tight_layout()

### Global maps, through the MCEq interface

The field below is read out of an MCEq atmosphere object built for each grid
node — not taken from the netCDF. That is the point: it exercises the same code
path a flux calculation uses, so what the map shows is what MCEq would
integrate.

**Vertical column depth** `atm.max_X` — the whole atmosphere overhead in
g/cm², measured from the ground, because every node is built with the surface
elevation from the table above. This is the overburden a vertical shower at
that point would actually traverse, so the map is dominated by orography: the
Himalaya, the Andes, the Antarctic plateau and Greenland stand out as deep
minima, with the synoptic pressure field as the finer structure over the ocean.

In [ ]:
def mceq_maps(table, elevation_m=None):
    """Vertical column depth above the ground at every grid node, via MCEq.

    One TabulatedAtmosphere per node, each told where its ground is, so the
    number is the overburden a shower there would actually traverse. It comes
    from max_X, i.e. from the same interface the solver integrates against.

    Args:
      table: gridded AtmosphereTable
      elevation_m: (n_lat, n_lon) surface elevation on *table*'s own axes,
        as returned by elevation_for(); recomputed when None

    Returns:
      (n_lat, n_lon) column depth in g/cm^2
    """
    if elevation_m is None:
        elevation_m = elevation_for(table)
    n_lat, n_lon = table.lat_deg.size, table.lon_deg.size
    max_X = np.empty((n_lat, n_lon))
    for j, node_lat in enumerate(table.lat_deg):
        for i, node_lon in enumerate(table.lon_deg):
            atm = dp.TabulatedAtmosphere(
                table,
                coord=(node_lon, node_lat),
                # MCEq's geometry measures altitude from sea level and will
                # not take a negative observation level, so the few basins
                # below it (Dead Sea, Caspian, Turfan) are clamped and come out
                # too shallow by the air they are missing -- about 6 g/cm^2 at
                # the -52 m this 4 deg grid reaches, and some 43 g/cm^2 at the
                # -360 m of the Dead Sea at ERA5's full resolution.
                surface_elevation_m=max(float(elevation_m[j, i]), 0.0),
            )
            atm.set_theta(0.0)
            max_X[j, i] = atm.max_X
    return max_X


import time

start = time.time()
max_X = mceq_maps(table, table_elevation_m)
print("%d columns through MCEq in %.1f s" % (max_X.size, time.time() - start))
print("column depth     %.1f .. %.1f g/cm^2" % (max_X.min(), max_X.max()))

In [ ]:
fig = plt.figure(figsize=(9, 5.0), dpi=120)
ax = fig.add_subplot(1, 1, 1, projection=ccrs.Robinson())
global_map(ax, max_X, table.lat_deg, table.lon_deg, "viridis",
           "vertical column depth max_X [g/cm$^2$]")
ax.plot(SITE_LON, SITE_LAT, "r*", ms=12, transform=ccrs.PlateCarree())
ax.set_title(f"ERA5 through MCEq — {str(ds['valid_time'].values[0])[:10]}")
fig.subplots_adjust(left=0.04, right=0.96, top=0.92)

#### A site on high ground

The map above starts every column at sea level. For a detector on the Antarctic
plateau that is 2.8 km of atmosphere that is not there — `surface_elevation_m`
is what removes it, and the difference is large enough to matter for any rate
calculation.

In [ ]:
SOUTH_POLE = (0.0, -90.0)  # lon, lat
ICECUBE_ELEVATION_M = 2835.0

sea_level = dp.TabulatedAtmosphere(table, coord=SOUTH_POLE, location="SouthPole")
on_the_ice = dp.TabulatedAtmosphere(
    table, coord=SOUTH_POLE, surface_elevation_m=ICECUBE_ELEVATION_M,
    location="SouthPole",
)
msis_ic = dp.MSIS00Atmosphere("SouthPole", "July")
for atm in (sea_level, on_the_ice, msis_ic):
    atm.set_theta(0.0)

print(f"ERA5, default (sea level)      {sea_level.max_X:7.2f} g/cm^2  "
      f"h_obs {sea_level.geom.h_obs / 1e5:.3f} km")
print(f"ERA5, surface_elevation_m set  {on_the_ice.max_X:7.2f} g/cm^2  "
      f"h_obs {on_the_ice.geom.h_obs / 1e5:.3f} km")
print(f"MSIS00 SouthPole               {msis_ic.max_X:7.2f} g/cm^2  "
      f"h_obs {msis_ic.geom.h_obs / 1e5:.3f} km")

#### Day to day

The same map for every day in the download, as the **anomaly** from the
five-day mean. The absolute field is not the useful thing to animate: it spans
about 100 g/cm² across the globe while a typical grid node moves less than
1 g/cm² from one day to the next, so on a common colour scale the animation
looks frozen. The departure from the mean is where the weather is — the
mid-latitude storm tracks swing by tens of g/cm², and that day-to-day motion of
the overburden is exactly what a seasonal-variation analysis has to fold in.

In [ ]:
from matplotlib import animation
from IPython.display import HTML

days = []
for path in table_files:
    day_table = dp.load_atmosphere_table(path)
    days.append(mceq_maps(day_table))
    print("done", path)

stack = np.array(days)
anomaly = stack - stack.mean(axis=0)
dates = [os.path.basename(p).replace("era5_", "").split(".")[0]
         for p in table_files]

# Symmetric about zero so the colour scale reads as a departure, not a value.
lim = np.percentile(np.abs(anomaly), 99)
print("column depth   %.1f .. %.1f g/cm^2 across the globe" % (stack.min(), stack.max()))
print("day-to-day     %.2f g/cm^2 for the median node, %.1f at the 99th pct"
      % (np.median(stack.max(0) - stack.min(0)), np.percentile(anomaly, 99)))

fig = plt.figure(figsize=(8, 4.5), dpi=110)
ax = fig.add_subplot(1, 1, 1, projection=ccrs.Robinson())
mesh = global_map(ax, anomaly[0], table.lat_deg, table.lon_deg, "RdBu_r",
                  r"max_X $-$ 5-day mean [g/cm$^2$]", vmin=-lim, vmax=lim)
ax.plot(SITE_LON, SITE_LAT, "k*", ms=11, transform=ccrs.PlateCarree())
title = ax.set_title("")


def draw(step):
    cyclic, _ = add_cyclic_point(anomaly[step], coord=table.lon_deg)
    mesh.set_array(cyclic.ravel())
    title.set_text(dates[step])
    return mesh, title


ani = animation.FuncAnimation(fig, draw, frames=len(dates), interval=700, blit=False)
plt.close(fig)
HTML(ani.to_jshtml())

### Why the grid matters for neutrinos

`TabulatedLocationCentered` follows the shower axis from the detector toward
the source and takes the table column where it crosses the surface. For a
downgoing shower that is near the detector. For an **upgoing** one — the signal
region of a neutrino telescope — the axis passes through the Earth and the
shower developed on the *far side*, in an atmosphere that has nothing to do
with the one overhead. A single-column table cannot express that; a global grid
can, which is why `max_theta=180` needs one.

The map below traces the impact point over the whole sky, coloured by the slant
depth MCEq computes there.

> **Open question — the elevation at the impact point.**
> The atmospheric *column* follows the impact point, but the *ground* under it
> does not. `TabulatedLocationCentered` uses one observation level for the
> whole sky: the detector's own elevation for downgoing showers, and sea level
> for the upgoing ones that develop on the far side. So a shower whose impact
> point lands on the Antarctic plateau or the Tibetan plateau is still
> integrated from the detector's elevation, and it picks up the couple of
> hundred g/cm² of air that the terrain there actually displaces.
>
> The surface table built at the top of this notebook is exactly what would fix
> it — one elevation per node, ready to be looked up at the impact point — but
> wiring it in raises questions that should be settled first rather than
> guessed at. Which elevation is even right for a shower that crosses a
> mountain range at 20° above the horizon, when the column is not vertical and
> the ground under it varies by kilometres along the path? At what zenith angle
> does the flat-Earth-under-the-impact-point picture stop being the dominant
> error compared with the curvature the geometry already handles? And should
> the surface elevation live in the atmosphere table itself, as another column
> per node, rather than in a table of its own?
>
> Until that is worked out, treat overburden for impact points over high
> terrain as approximate. It does not affect the vertical maps above, which set
> each node's elevation explicitly.

In [ ]:
lc_atm = dp.TabulatedLocationCentered(
    table,
    detector_coord=(SITE_LON, SITE_LAT),
    depth_m=SITE_DEPTH_M,
    max_theta=180.0,
    location=SITE_NAME,
    season="July",
)

zeniths = np.arange(0.0, 180.1, 7.5)
azimuths = np.arange(0.0, 360.0, 45.0)
track = []
for azimuth in azimuths:
    for zenith in zeniths:
        lc_atm.set_theta(float(zenith), azimuth_deg=float(azimuth))
        track.append(
            (lc_atm.current_impact_longitude, lc_atm.current_impact_latitude,
             lc_atm.max_X, zenith, azimuth)
        )
track = np.array(track)
print(f"{len(track)} directions, slant depth "
      f"{track[:, 2].min():.0f} .. {track[:, 2].max():.0f} g/cm^2")

In [ ]:
fig = plt.figure(figsize=(13, 5.2), dpi=120)
downgoing = track[:, 3] <= 90.0

# --- the whole sky ---------------------------------------------------------
ax = fig.add_subplot(1, 2, 1, projection=ccrs.Robinson())
global_map(ax, max_X, table.lat_deg, table.lon_deg, paled("viridis"),
           "vertical column depth max_X [g/cm$^2$]")
for mask, marker, name in (
    (downgoing, "o", "downgoing"),
    (~downgoing, "^", "upgoing (far side)"),
):
    scat = ax.scatter(
        track[mask, 0], track[mask, 1], c=np.log10(track[mask, 2]),
        s=22, marker=marker, cmap="viridis", vmin=np.log10(track[:, 2]).min(),
        vmax=np.log10(track[:, 2]).max(), transform=ccrs.PlateCarree(),
        zorder=3, edgecolors="k", linewidths=0.25, label=name,
    )
ax.plot(SITE_LON, SITE_LAT, "r*", ms=15, transform=ccrs.PlateCarree(), zorder=4)
plt.colorbar(scat, ax=ax, orientation="vertical", pad=0.02,
             label=r"$\log_{10}$ slant depth [g/cm$^2$]")
ax.legend(loc="lower left", fontsize=8)
ax.set_title(f"All sky from {SITE_NAME} (star):\nupgoing showers develop on the far side", fontsize=10)

# --- and the downgoing cluster, which the star hides above ------------------
ax2 = fig.add_subplot(1, 2, 2, projection=ccrs.PlateCarree())
near = ax2.scatter(
    track[downgoing, 0], track[downgoing, 1], c=track[downgoing, 3],
    s=26, cmap="viridis", transform=ccrs.PlateCarree(), zorder=3,
    edgecolors="k", linewidths=0.25,
)
ax2.plot(SITE_LON, SITE_LAT, "r*", ms=15, transform=ccrs.PlateCarree(), zorder=4)
ax2.set_extent([SITE_LON - 3.5, SITE_LON + 3.5, SITE_LAT - 3.0, SITE_LAT + 3.0],
               crs=ccrs.PlateCarree())
ax2.coastlines(lw=0.5, color="0.3")
grid = ax2.gridlines(draw_labels=True, lw=0.3, color="0.7")
grid.top_labels = grid.right_labels = False
plt.colorbar(near, ax=ax2, orientation="vertical", pad=0.02,
             label="zenith angle [deg]")
ax2.set_title("Downgoing: the column drifts\nwith zenith and azimuth", fontsize=10)

# tight_layout does not cope with cartopy gridline labels
fig.subplots_adjust(left=0.02, right=0.97, top=0.86, wspace=0.28)

### Where to go from here

* **Humidity.** Everything above is dry air. `specific_humidity` is on the same
  pressure levels; folding it in changes the density by less than a percent
  near the surface and less above, so it matters only for precision work.
* **Resolution.** `STRIDE = 4` is 1 deg, about 110 km, and the impact point
  moves ~100 km by 80 deg zenith — so a global run is sampling the atmosphere
  about as finely as the shower geometry needs. What stops you going to the
  native 0.25 deg is the table on disk, not MCEq: 38 M rows per day, 1.8 GB of
  CSV or 480 MB gzipped. For a regional analysis crop first, then keep every
  point.
* **Seasonal studies.** One table per day; build one atmosphere per table and
  pass them to `MCEqRun.solve_batch(..., conditions=...)` as per-member
  `density_model`s to get a time series in one call.
* **Other datasets.** Only the converter above is ERA5-specific. Write
  `h_cm` plus `T_K`/`p_hPa` (or `rho_gcm3`), optionally `lat_deg`/`lon_deg`,
  and any archive works.